# Loading

In [ ]:
from plot_grn import create_network_from_dict,grn_network_from_dict
import omicverse as ov
import pandas as pd
import json
import numpy as np
from adjustText import adjust_text
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
ov.plot_set(font_path='Arial')
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

In [ ]:
def split_long_name(name, threshold=50):
    if len(name) > threshold:
        middle = len(name) // 2
        split_point = middle
        while split_point > 0 and name[split_point] != ' ':
            split_point -= 1
        if split_point > 0:  
            return name[:split_point] + '\n' + name[split_point+1:]
        else:
            return name
    else:
        return name
        
def pathway_enrichment(enr,
                      figsize=(6, 6),
                      color='#aec7e8',
                      threshold=50,fontsize=14,
                      linewidth=1.5,
                      title=''):
    """
    Plot the enrichment results with split long names.

    Parameters:
        enr: DataFrame containing the enrichment results with columns 'Term' and 'Adjusted P-value'.
        figsize: Tuple specifying the figure size (width, height).
        color: String specifying the color of the bars.
        threshold: Integer specifying the maximum length of a term before it is split.
        linewidth: Float specifying the line width of the horizontal line.
        title: String specifying the title of the plot.
    """
    enr['Term'] = enr['Term'].str.replace('\(GO:\d+\)', '', regex=True)
    enr['-log10(Adjusted P-value)'] = -np.log10(enr['Adjusted P-value'])
    enr_sorted = enr.sort_values('-log10(Adjusted P-value)', ascending=False)
    enr_sorted['Term'] = enr_sorted['Term'].apply(lambda x: split_long_name(x, threshold))

    f, ax = plt.subplots(1, 1, figsize=figsize, sharex=True) 
    colors = [color] * len(enr.index)
    barplot = sns.barplot(x="-log10(Adjusted P-value)",
                          y="Term", data=enr_sorted, palette=colors, ax=ax)


    for i, p in enumerate(ax.patches):  
        ax.text(p.get_x()+0.1, p.get_y() + p.get_height() / 2. + 0.05,
                f'{enr_sorted["Term"].iloc[i]}', fontsize=fontsize,
                ha='left', va='center', color='black')

    ax.set_title(title,fontweight='bold',fontsize=fontsize+1)
    ax.set_ylabel('')

    ax.spines['bottom'].set_visible(True)
    ax.spines['bottom'].set_linewidth(linewidth)
    ax.spines['left'].set_visible(True)
    ax.spines['left'].set_linewidth(linewidth)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    ax.set_xlabel('-log$_{10}$ Adjusted $\it{P}$',fontsize=16,)
    plt.setp(ax.get_yticklabels(), visible=False)
    plt.tight_layout(h_pad=2)
    plt.grid(False)

    return f,ax

def top_n_unique_genes(pathway_to_genes, df, gene_col="gene", expr_col="mean_counts",n=5):
    all_genes = []
    for pw, genes in pathway_to_genes.items():
        tmp = df[df[gene_col].isin(genes)].sort_values(expr_col, ascending=False)
        all_genes.extend(tmp[gene_col].head(7).tolist())
    seen = set()
    unique_genes = []
    for g in all_genes:
        if g not in seen:
            seen.add(g)
            unique_genes.append(g)
    return unique_genes

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import omicverse as ov 

def run_grn_analysis_plot(keys, grn_dict, pathway_dict, adata_var, save_prefix, other_target_size=10,specific_target_size=150,threshold=30, ):
    # 确保保存目录存在
    save_dir = os.path.dirname(save_prefix)
    if save_dir and not os.path.exists(save_dir):
        os.makedirs(save_dir)

    # 1. 数据准备
    grn_dict_sub = {k: grn_dict[k] for k in keys if k in grn_dict}
    s = pd.Series(grn_dict_sub)
    df = s.explode().reset_index()
    df.columns = ['TF', 'Target Gene']

    # 2. 通路富集分析
    enr = ov.bulk.geneset_enrichment(gene_list=list(set(df['Target Gene'])),
                                     pathways_dict=pathway_dict,
                                     pvalue_type='auto',
                                     organism='Human')

    pathway_list = enr.sort_values(['Adjusted P-value'])['Term'][:5].tolist()
    subset = enr[enr['Term'].isin(pathway_list)]

    pathway_to_genes = {}
    for index, row in subset.iterrows():
        pathway_to_genes[row['Term']] = [g.strip() for g in row['Genes'].split(';') if g.strip()]

    enr = enr[enr['Term'].isin(pathway_to_genes.keys())]

    # 3. 绘制并保存富集图
    fig, ax = pathway_enrichment(enr.sort_values(['Adjusted P-value'], ascending=True).iloc[0:5, :],
                                 color='#B0D8E8', fontsize=15, figsize=(6, 4), threshold=threshold, title='')
    ax.set_xlabel('-log$_{10}$ Adjusted $\it{P}$', fontsize=16)
    plt.grid(False)
    ax.tick_params(axis='x', which='major', labelsize=16, width=1.5, length=4, direction='out')
    ax.tick_params(axis='y', which='major', labelsize=16, width=1.5, length=0, direction='out')
    
    # 保存路径参考：Figure/HIP/HIP_sc19_selected_GRN_pathway.pdf
    fig.savefig(f"{save_prefix}_selected_GRN_pathway.pdf", dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig) # 关闭画布防止重叠

    # 4. 绘制并保存网络图
    G, G_type, G_color = create_network_from_dict(grn_dict_sub, n_top=5000, 
                                                  color_dict={'tf_color': '#7030a0', 'target_color': '#ffc000'})
    
    fig, ax = grn_network_from_dict(
        G, G_type, G_color, grn_dict_sub,
        figsize=(8, 6.5), pos_scale=7, TF_size=300,
        node_alpha=0.8, node_linewidths=0.5,
        TF_fontsize=12, target_fontsize=11,
        specific_gene=top_n_unique_genes(pathway_to_genes, adata_var),
        specific_target_size=specific_target_size,
        other_target_size=other_target_size
    )
    
    # 自动从 save_prefix 提取标题名 (例如从 Figure/HIP/HIP_sc20 提取 HIP_sc20)
    title_name = os.path.basename(save_prefix)
    plt.title(f'{title_name} GRN', fontsize=15, weight='bold')
    
    # 保存路径参考：Figure/HIP/HIP_sc20_GRN.pdf
    fig.savefig(f"{save_prefix}_GRN.pdf", dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
pathway_dict=ov.utils.geneset_prepare('genesets/GO_Biological_Process_2021.txt',organism='Human')

# midbrain

In [ ]:
adata = sc.read_h5ad("Process_Data/bin100_3D_region/combined_adata_midbrain.h5ad")
sc.pp.calculate_qc_metrics(adata, qc_vars=[], inplace=True, log1p=True)
adata_var = adata.var
adata_var['gene'] = adata_var.index
del adata

In [ ]:
with open('Output/GRN_GPU_output/midbrain/tmp_files/regulons.json', 'r', encoding='utf-8') as file:
    grn_dict = json.load(file)

In [ ]:
keys = ['FOXA2(+)','LMX1(+)','EN2(+)','PITX1(+)','PAX5(+)','RORB(+)','NR2F2(+)']
grn_dict_sub = {k: grn_dict[k] for k in keys if k in grn_dict}

s = pd.Series(grn_dict_sub)
s_exploded = s.explode()
df = s_exploded.reset_index()
df.columns = ['TF', 'Target Gene']
df.to_csv("Output/whole_brain_selected_GRN/midbrain_selected_GRN.csv")

In [ ]:
enr=ov.bulk.geneset_enrichment(gene_list=list(set(df['Target Gene'])),
                                pathways_dict=pathway_dict,
                                pvalue_type='auto',
                                organism='Human')

pathway_1 = 'chemical synaptic transmission (GO:0007268)'
pathway_2 = 'anterograde trans-synaptic signaling (GO:0098916)'
pathway_3 = 'axonogenesis (GO:0007409)'
pathway_4 = 'signal release from synapse (GO:0099643)'
pathway_5 = 'synapse organization (GO:0050808)'

pathway_to_genes = {pathway_1:[g.strip() for g in list(enr.loc[enr['Term'] == pathway_1, 'Genes'].values)[0].split(';') if g.strip()],
                    pathway_2:[g.strip() for g in list(enr.loc[enr['Term'] == pathway_2, 'Genes'].values)[0].split(';') if g.strip()],
                    pathway_3:[g.strip() for g in list(enr.loc[enr['Term'] == pathway_3, 'Genes'].values)[0].split(';') if g.strip()],
                    pathway_4:[g.strip() for g in list(enr.loc[enr['Term'] == pathway_3, 'Genes'].values)[0].split(';') if g.strip()],
                    pathway_5:[g.strip() for g in list(enr.loc[enr['Term'] == pathway_3, 'Genes'].values)[0].split(';') if g.strip()],}


target_pathway = pathway_to_genes.keys()
enr = enr[enr['Term'].isin(target_pathway)]

fig,ax = pathway_enrichment(enr,color='#B0D8E8',fontsize=15
                                  ,figsize=(6,5),threshold=40,
                                  title='')
ax.set_xlabel('-log$_{10}$ Adjusted $\it{P}$',fontsize=16,)
plt.grid(False)
ax.tick_params(axis='x', which='major', labelsize=16, width=1.5, length=4,direction='out')
ax.tick_params(axis='y', which='major', labelsize=16, width=1.5, length=0,direction='out')
fig.savefig("Figure/whole_brain_selected_GRN/Pathway/midbrain_pathway.pdf", dpi=300, bbox_inches = "tight")
enr.sort_values(['Adjusted P-value']).to_csv("Output/whole_brain_selected_GRN/0_midbrain_selected_GRN_pathway.csv")

In [ ]:
import matplotlib.pyplot as plt
G, G_type, G_color = create_network_from_dict(grn_dict_sub, n_top=5000,color_dict={'tf_color':'#7030a0','target_color':'#ffc000'})
fig, ax = grn_network_from_dict(
    G,
    G_type,
    G_color,
    grn_dict_sub,
    figsize=(8,7),
    pos_scale=7,
    TF_size=200,
    node_alpha=0.8,
    node_linewidths=0.5,
    TF_fontsize=12,
    target_fontsize=12,
    specific_gene=top_n_unique_genes(pathway_to_genes,adata_var),
    specific_target_size=100,                  # 指定靶基因节点大小
    other_target_size=10                       # 其他靶基因节点大小
)
plt.title('midbrain GRN',fontsize=15,weight='bold',)
fig.savefig("Figure/whole_brain_selected_GRN/GRN/midbrain_GRN.pdf", dpi=300, bbox_inches = "tight")
plt.show()

# HIP

In [ ]:
adata = sc.read_h5ad("Process_Data/bin100_3D_region/combined_adata_HIP.h5ad")
sc.pp.calculate_qc_metrics(adata, qc_vars=[], inplace=True, log1p=True)
adata_var = adata.var
adata_var['gene'] = adata_var.index
del adata

In [ ]:
with open('Output/GRN_GPU_output/HIP/tmp_files/regulons.json', 'r', encoding='utf-8') as file:
    grn_dict = json.load(file)

In [ ]:
keys = ['LHX1(+)','HDAC2(+)','SOX8(+)','ZBTB11(+)']
grn_dict_sub = {k: grn_dict[k] for k in keys if k in grn_dict}

s = pd.Series(grn_dict_sub)
s_exploded = s.explode()
df = s_exploded.reset_index()
df.columns = ['TF', 'Target Gene']
df.to_csv("Output/whole_brain_selected_GRN/HIP_selected_GRN.csv")

In [ ]:
enr=ov.bulk.geneset_enrichment(gene_list=list(set(df['Target Gene'])),
                                pathways_dict=pathway_dict,
                                pvalue_type='auto',
                                organism='Human')

pathway_1 = 'pyrimidine nucleobase catabolic process (GO:0006208)'
pathway_2 = 'nucleobase catabolic process (GO:0046113)'
pathway_3 = 'regulation of apoptotic process (GO:0042981)'
pathway_4 = 'nervous system development (GO:0007399)'

pathway_to_genes = {pathway_1:[g.strip() for g in list(enr.loc[enr['Term'] == pathway_1, 'Genes'].values)[0].split(';') if g.strip()],
                    pathway_2:[g.strip() for g in list(enr.loc[enr['Term'] == pathway_2, 'Genes'].values)[0].split(';') if g.strip()],
                    pathway_3:[g.strip() for g in list(enr.loc[enr['Term'] == pathway_3, 'Genes'].values)[0].split(';') if g.strip()],
                    pathway_4:[g.strip() for g in list(enr.loc[enr['Term'] == pathway_4, 'Genes'].values)[0].split(';') if g.strip()]}

pathway_to_genes

In [ ]:
import matplotlib.pyplot as plt
G, G_type, G_color = create_network_from_dict(grn_dict_sub, n_top=5000,color_dict={'tf_color':'#7030a0','target_color':'#ffc000'})
fig, ax = grn_network_from_dict(
    G,
    G_type,
    G_color,
    grn_dict_sub,
    figsize=(8,7),
    pos_scale=7,
    TF_size=300,
    node_alpha=0.8,
    node_linewidths=0.5,
    TF_fontsize=12,
    target_fontsize=12,
    specific_gene=top_n_unique_genes(pathway_to_genes,adata_var),
    specific_target_size=150,                  # 指定靶基因节点大小
    other_target_size=20                       # 其他靶基因节点大小
)
plt.title('HIP GRN',fontsize=15,weight='bold',)
fig.savefig("Figure/whole_brain_selected_GRN/GRN/HIP_GRN.pdf", dpi=300, bbox_inches = "tight")
plt.show()

# VZ

In [ ]:
adata = sc.read_h5ad("Process_Data/bin100_3D_region/combined_adata_VZ.h5ad")
sc.pp.calculate_qc_metrics(adata, qc_vars=[], inplace=True, log1p=True)
adata_var = adata.var
adata_var['gene'] = adata_var.index
del adata

with open('Output/GRN_GPU_output/VZ/tmp_files/regulons.json', 'r', encoding='utf-8') as file:
    grn_dict = json.load(file)

In [ ]:
keys = ['SP8(+)','E2F2(+)','E2F6(+)','CHD1(+)']
grn_dict_sub = {k: grn_dict[k] for k in keys if k in grn_dict}

s = pd.Series(grn_dict_sub)
s_exploded = s.explode()
df = s_exploded.reset_index()
df.columns = ['TF', 'Target Gene']
df.to_csv("Output/whole_brain_selected_GRN/VZ_selected_GRN.csv")
df

In [ ]:
enr=ov.bulk.geneset_enrichment(gene_list=list(set(df['Target Gene'])),
                                pathways_dict=pathway_dict,
                                pvalue_type='auto',
                                organism='Human')

pathway_1 = 'cellular protein modification process (GO:0006464)'
pathway_2 = 'mitotic spindle organization (GO:0007052)'
pathway_3 = 'mitotic cell cycle phase transition (GO:0044772)'
pathway_4 = 'regulation of mitotic cell cycle phase transition (GO:1901990)'
pathway_5 = 'mRNA splicing, via spliceosome (GO:0000398)'

pathway_to_genes = {pathway_1:[g.strip() for g in list(enr.loc[enr['Term'] == pathway_1, 'Genes'].values)[0].split(';') if g.strip()],
                    pathway_2:[g.strip() for g in list(enr.loc[enr['Term'] == pathway_2, 'Genes'].values)[0].split(';') if g.strip()],
                    pathway_3:[g.strip() for g in list(enr.loc[enr['Term'] == pathway_3, 'Genes'].values)[0].split(';') if g.strip()],
                    pathway_4:[g.strip() for g in list(enr.loc[enr['Term'] == pathway_3, 'Genes'].values)[0].split(';') if g.strip()],
                    pathway_5:[g.strip() for g in list(enr.loc[enr['Term'] == pathway_3, 'Genes'].values)[0].split(';') if g.strip()],}


target_pathway = pathway_to_genes.keys()
enr = enr[enr['Term'].isin(target_pathway)]

fig,ax = pathway_enrichment(enr,color='#B0D8E8',fontsize=15
                                  ,figsize=(6,5),threshold=40,
                                  title='')
ax.set_xlabel('-log$_{10}$ Adjusted $\it{P}$',fontsize=16,)
plt.grid(False)
ax.tick_params(axis='x', which='major', labelsize=16, width=1.5, length=4,direction='out')
ax.tick_params(axis='y', which='major', labelsize=16, width=1.5, length=0,direction='out')
fig.savefig("Figure/whole_brain_selected_GRN/Pathway/VZ_pathway.pdf", dpi=300, bbox_inches = "tight")
enr.sort_values(['Adjusted P-value']).to_csv("Output/whole_brain_selected_GRN/0_VZ_selected_GRN_pathway.csv")

In [ ]:
import matplotlib.pyplot as plt
G, G_type, G_color = create_network_from_dict(grn_dict_sub, n_top=5000,color_dict={'tf_color':'#7030a0','target_color':'#ffc000'})
fig, ax = grn_network_from_dict(
    G,
    G_type,
    G_color,
    grn_dict_sub,
    figsize=(8,7),
    pos_scale=7,
    TF_size=200,
    node_alpha=0.8,
    node_linewidths=0.5,
    TF_fontsize=12,
    target_fontsize=12,
    specific_gene=top_n_unique_genes(pathway_to_genes,adata_var),
    specific_target_size=100,                  # 指定靶基因节点大小
    other_target_size=5                       # 其他靶基因节点大小
)
plt.title('VZ GRN',fontsize=15,weight='bold',)
fig.savefig("Figure/whole_brain_selected_GRN/GRN/VZ_GRN.pdf", dpi=300, bbox_inches = "tight")
plt.show()

# SVZ

In [ ]:
adata = sc.read_h5ad("Process_Data/bin100_3D_region/combined_adata_SVZ.h5ad")
sc.pp.calculate_qc_metrics(adata, qc_vars=[], inplace=True, log1p=True)
adata_var = adata.var
adata_var['gene'] = adata_var.index
del adata

with open('Output/GRN_GPU_output/SVZ/tmp_files/regulons.json', 'r', encoding='utf-8') as file:
    grn_dict = json.load(file)

In [ ]:
keys = ['SMAD9(+)','EOMES(+)','SOX10(+)','TEAD1(+)','NHLH1(+)','SMAD7(+)']
grn_dict_sub = {k: grn_dict[k] for k in keys if k in grn_dict}

s = pd.Series(grn_dict_sub)
s_exploded = s.explode()
df = s_exploded.reset_index()
df.columns = ['TF', 'Target Gene']
df.to_csv("Output/whole_brain_selected_GRN/SVZ_selected_GRN.csv")
df

In [ ]:
enr=ov.bulk.geneset_enrichment(gene_list=list(set(df['Target Gene'])),
                                pathways_dict=pathway_dict,
                                pvalue_type='auto',
                                organism='Human')

pathway_1 = 'mRNA splicing, via spliceosome (GO:0000398)'
pathway_2 = 'regulation of mitotic cell cycle phase transition (GO:1901990)'
pathway_3 = 'mitotic spindle organization (GO:0007052)'
pathway_4 = 'regulation of G2/M transition of mitotic cell cycle (GO:0010389)'
pathway_5 = 'DNA metabolic process (GO:0006259)'

pathway_to_genes = {pathway_1:[g.strip() for g in list(enr.loc[enr['Term'] == pathway_1, 'Genes'].values)[0].split(';') if g.strip()],
                    pathway_2:[g.strip() for g in list(enr.loc[enr['Term'] == pathway_2, 'Genes'].values)[0].split(';') if g.strip()],
                    pathway_3:[g.strip() for g in list(enr.loc[enr['Term'] == pathway_3, 'Genes'].values)[0].split(';') if g.strip()],
                    pathway_4:[g.strip() for g in list(enr.loc[enr['Term'] == pathway_3, 'Genes'].values)[0].split(';') if g.strip()],
                    pathway_5:[g.strip() for g in list(enr.loc[enr['Term'] == pathway_3, 'Genes'].values)[0].split(';') if g.strip()],}


target_pathway = pathway_to_genes.keys()
enr = enr[enr['Term'].isin(target_pathway)]

fig,ax = pathway_enrichment(enr,color='#B0D8E8',fontsize=15
                                  ,figsize=(6,5),threshold=40,
                                  title='')
ax.set_xlabel('-log$_{10}$ Adjusted $\it{P}$',fontsize=16,)
plt.grid(False)
ax.tick_params(axis='x', which='major', labelsize=16, width=1.5, length=4,direction='out')
ax.tick_params(axis='y', which='major', labelsize=16, width=1.5, length=0,direction='out')
fig.savefig("Figure/whole_brain_selected_GRN/Pathway/SVZ_pathway.pdf", dpi=300, bbox_inches = "tight")
enr.sort_values(['Adjusted P-value']).to_csv("Output/whole_brain_selected_GRN/0_SVZ_selected_GRN_pathway.csv")

In [ ]:
import matplotlib.pyplot as plt
G, G_type, G_color = create_network_from_dict(grn_dict_sub, n_top=5000,color_dict={'tf_color':'#7030a0','target_color':'#ffc000'})
fig, ax = grn_network_from_dict(
    G,
    G_type,
    G_color,
    grn_dict_sub,
    figsize=(8,7),
    pos_scale=7,
    TF_size=200,
    node_alpha=0.8,
    node_linewidths=0.5,
    TF_fontsize=12,
    target_fontsize=12,
    specific_gene=top_n_unique_genes(pathway_to_genes,adata_var),
    specific_target_size=100,                  # 指定靶基因节点大小
    other_target_size=5                       # 其他靶基因节点大小
)
plt.title('SVZ GRN',fontsize=15,weight='bold',)
fig.savefig("Figure/whole_brain_selected_GRN/GRN/SVZ_GRN.pdf", dpi=300, bbox_inches = "tight")
plt.show()

# subthalamic_nucleus

In [ ]:
adata = sc.read_h5ad("Process_Data/bin100_3D_region/combined_adata_subthalamic nucleus.h5ad")
sc.pp.calculate_qc_metrics(adata, qc_vars=[], inplace=True, log1p=True)
adata_var = adata.var
adata_var['gene'] = adata_var.index
del adata

with open('Output/GRN_GPU_output/subthalamic_nucleus/tmp_files/regulons.json', 'r', encoding='utf-8') as file:
    grn_dict = json.load(file)

In [ ]:
keys = ['DDIT3(+)','RUNX3(+)','RCOR1(+)','HIC1(+)',]
grn_dict_sub = {k: grn_dict[k] for k in keys if k in grn_dict}

s = pd.Series(grn_dict_sub)
s_exploded = s.explode()
df = s_exploded.reset_index()
df.columns = ['TF', 'Target Gene']
df.to_csv("Output/whole_brain_selected_GRN/subthalamic_nucleus_selected_GRN.csv")
df

In [ ]:
enr=ov.bulk.geneset_enrichment(gene_list=list(set(df['Target Gene'])),
                                pathways_dict=pathway_dict,
                                pvalue_type='auto',
                                organism='Human')

pathway_1 = 'chemical synaptic transmission (GO:0007268)'
pathway_2 = 'phosphorylation (GO:0016310)'
pathway_3 = 'axonogenesis (GO:0007409)'
pathway_4 = 'regulation of dopamine secretion (GO:0014059)'

pathway_to_genes = {pathway_1:[g.strip() for g in list(enr.loc[enr['Term'] == pathway_1, 'Genes'].values)[0].split(';') if g.strip()],
                    pathway_2:[g.strip() for g in list(enr.loc[enr['Term'] == pathway_2, 'Genes'].values)[0].split(';') if g.strip()],
                    pathway_3:[g.strip() for g in list(enr.loc[enr['Term'] == pathway_3, 'Genes'].values)[0].split(';') if g.strip()],
                    pathway_4:[g.strip() for g in list(enr.loc[enr['Term'] == pathway_4, 'Genes'].values)[0].split(';') if g.strip()],}

In [ ]:
target_pathway = pathway_to_genes.keys()
enr = enr[enr['Term'].isin(target_pathway)]

fig,ax = pathway_enrichment(enr.sort_values(['Adjusted P-value'],ascending=True).iloc[0:4,:],color='#FBCAC8',fontsize=15
                                  ,figsize=(4.5,3.5),threshold=40,
                                  title='')
ax.set_xlabel('-log$_{10}$ Adjusted $\it{P}$',fontsize=16,)
plt.grid(False)
ax.tick_params(axis='x', which='major', labelsize=16, width=1.5, length=4,direction='out')
ax.tick_params(axis='y', which='major', labelsize=16, width=1.5, length=0,direction='out')
fig.savefig("Figure/whole_brain_selected_GRN/Pathway/subthalamic_nucleus_pathway.pdf", dpi=300, bbox_inches = "tight")
enr.sort_values(['Adjusted P-value']).to_csv("Output/whole_brain_selected_GRN/0_subthalamic_nucleus_selected_GRN_pathway.csv")

In [ ]:
import matplotlib.pyplot as plt
G, G_type, G_color = create_network_from_dict(grn_dict_sub, n_top=5000,color_dict={'tf_color':'#7030a0','target_color':'#ffc000'})
fig, ax = grn_network_from_dict(
    G,
    G_type,
    G_color,
    grn_dict_sub,
    figsize=(8,7),
    pos_scale=7,
    TF_size=200,
    node_alpha=0.8,
    node_linewidths=0.5,
    TF_fontsize=12,
    target_fontsize=12,
    specific_gene=top10_unique_genes(pathway_to_genes,adata_var),
    specific_target_size=100,                  # 指定靶基因节点大小
    other_target_size=10                       # 其他靶基因节点大小
)
plt.title('subthalamic nucleus GRN',fontsize=15,weight='bold',)
fig.savefig("Figure/whole_brain_selected_GRN/GRN/subthalamic_nucleus_GRN.pdf", dpi=300, bbox_inches = "tight")
plt.show()

# CP/SP

In [ ]:
adata = sc.read_h5ad("Process_Data/bin100_3D_region/combined_adata_CP_SP.h5ad")
sc.pp.calculate_qc_metrics(adata, qc_vars=[], inplace=True, log1p=True)
adata_var = adata.var
adata_var['gene'] = adata_var.index
del adata

with open('Output/GRN_GPU_output/CP_SP/tmp_files/regulons.json', 'r', encoding='utf-8') as file:
    grn_dict = json.load(file)

In [ ]:
keys = ['ARID3A(+)','BCL6(+)','PBX1(+)','POU3F2(+)','SOX18(+)']
grn_dict_sub = {k: grn_dict[k] for k in keys if k in grn_dict}

s = pd.Series(grn_dict_sub)
s_exploded = s.explode()
df = s_exploded.reset_index()
df.columns = ['TF', 'Target Gene']
df.to_csv("Output/whole_brain_selected_GRN/CP_SP_selected_GRN.csv")
df

In [ ]:
enr=ov.bulk.geneset_enrichment(gene_list=list(set(df['Target Gene'])),
                                pathways_dict=pathway_dict,
                                pvalue_type='auto',
                                organism='Human')

pathway_1 = 'cellular protein modification process (GO:0006464)'
pathway_2 = 'axonogenesis (GO:0007409)'
pathway_3 = 'neuron projection development (GO:0031175)'
pathway_4 = 'protein ubiquitination (GO:0016567)'
pathway_5 = 'nervous system development (GO:0007399)'

pathway_to_genes = {pathway_1:[g.strip() for g in list(enr.loc[enr['Term'] == pathway_1, 'Genes'].values)[0].split(';') if g.strip()],
                    pathway_2:[g.strip() for g in list(enr.loc[enr['Term'] == pathway_2, 'Genes'].values)[0].split(';') if g.strip()],
                    pathway_3:[g.strip() for g in list(enr.loc[enr['Term'] == pathway_3, 'Genes'].values)[0].split(';') if g.strip()],
                    pathway_4:[g.strip() for g in list(enr.loc[enr['Term'] == pathway_3, 'Genes'].values)[0].split(';') if g.strip()],
                    pathway_5:[g.strip() for g in list(enr.loc[enr['Term'] == pathway_3, 'Genes'].values)[0].split(';') if g.strip()],}


target_pathway = pathway_to_genes.keys()
enr = enr[enr['Term'].isin(target_pathway)]

fig,ax = pathway_enrichment(enr,color='#B0D8E8',fontsize=15
                                  ,figsize=(6,5),threshold=40,
                                  title='')
ax.set_xlabel('-log$_{10}$ Adjusted $\it{P}$',fontsize=16,)
plt.grid(False)
ax.tick_params(axis='x', which='major', labelsize=16, width=1.5, length=4,direction='out')
ax.tick_params(axis='y', which='major', labelsize=16, width=1.5, length=0,direction='out')
fig.savefig("Figure/whole_brain_selected_GRN/Pathway/CP_SP_pathway.pdf", dpi=300, bbox_inches = "tight")
enr.sort_values(['Adjusted P-value']).to_csv("Output/whole_brain_selected_GRN/0_CP_SP_selected_GRN_pathway.csv")

In [ ]:
import matplotlib.pyplot as plt
G, G_type, G_color = create_network_from_dict(grn_dict_sub, n_top=5000,color_dict={'tf_color':'#7030a0','target_color':'#ffc000'})
fig, ax = grn_network_from_dict(
    G,
    G_type,
    G_color,
    grn_dict_sub,
    figsize=(8,7),
    pos_scale=7,
    TF_size=200,
    node_alpha=0.8,
    node_linewidths=0.5,
    TF_fontsize=12,
    target_fontsize=12,
    specific_gene=top_n_unique_genes(pathway_to_genes,adata_var),
    specific_target_size=100,                  # 指定靶基因节点大小
    other_target_size=10                      # 其他靶基因节点大小
)
plt.title('CP/SP GRN',fontsize=15,weight='bold',)
fig.savefig("Figure/whole_brain_selected_GRN/GRN/CP_SP_GRN.pdf", dpi=300, bbox_inches = "tight")
plt.show()

# thalamus

In [ ]:
adata = sc.read_h5ad("Process_Data/bin100_3D_region/combined_adata_thalamus.h5ad")
sc.pp.calculate_qc_metrics(adata, qc_vars=[], inplace=True, log1p=True)
adata_var = adata.var
adata_var['gene'] = adata_var.index
del adata

with open('Output/GRN_GPU_output/thalamus/tmp_files/regulons.json', 'r', encoding='utf-8') as file:
    grn_dict = json.load(file)

In [ ]:
keys = ['NR1H2(+)','POU6F1(+)','LIN28B(+)','KDM5A(+)','DLX5(+)']
grn_dict_sub = {k: grn_dict[k] for k in keys if k in grn_dict}

s = pd.Series(grn_dict_sub)
s_exploded = s.explode()
df = s_exploded.reset_index()
df.columns = ['TF', 'Target Gene']
df.to_csv("Output/whole_brain_selected_GRN/thalamus_selected_GRN.csv")
df

In [ ]:
enr=ov.bulk.geneset_enrichment(gene_list=list(set(df['Target Gene'])),
                                pathways_dict=pathway_dict,
                                pvalue_type='auto',
                                organism='Human')

pathway_1 = 'axonogenesis (GO:0007409)'
pathway_2 = 'regulation of neuron projection development (GO:0010975)'
pathway_3 = 'axon guidance (GO:0007411)'
pathway_4 = 'neuron projection development (GO:0031175)'
pathway_5 = 'chemical synaptic transmission (GO:0007268)'

pathway_to_genes = {pathway_1:[g.strip() for g in list(enr.loc[enr['Term'] == pathway_1, 'Genes'].values)[0].split(';') if g.strip()],
                    pathway_2:[g.strip() for g in list(enr.loc[enr['Term'] == pathway_2, 'Genes'].values)[0].split(';') if g.strip()],
                    pathway_3:[g.strip() for g in list(enr.loc[enr['Term'] == pathway_3, 'Genes'].values)[0].split(';') if g.strip()],
                    pathway_4:[g.strip() for g in list(enr.loc[enr['Term'] == pathway_3, 'Genes'].values)[0].split(';') if g.strip()],
                    pathway_5:[g.strip() for g in list(enr.loc[enr['Term'] == pathway_3, 'Genes'].values)[0].split(';') if g.strip()],}


target_pathway = pathway_to_genes.keys()
enr = enr[enr['Term'].isin(target_pathway)]

fig,ax = pathway_enrichment(enr,color='#B0D8E8',fontsize=15
                                  ,figsize=(6,5),threshold=40,
                                  title='')
ax.set_xlabel('-log$_{10}$ Adjusted $\it{P}$',fontsize=16,)
plt.grid(False)
ax.tick_params(axis='x', which='major', labelsize=16, width=1.5, length=4,direction='out')
ax.tick_params(axis='y', which='major', labelsize=16, width=1.5, length=0,direction='out')
fig.savefig("Figure/whole_brain_selected_GRN/Pathway/thalamus_pathway.pdf", dpi=300, bbox_inches = "tight")
enr.sort_values(['Adjusted P-value']).to_csv("Output/whole_brain_selected_GRN/0_thalamus_selected_GRN_pathway.csv")

In [ ]:
import matplotlib.pyplot as plt
G, G_type, G_color = create_network_from_dict(grn_dict_sub, n_top=5000,color_dict={'tf_color':'#7030a0','target_color':'#ffc000'})
fig, ax = grn_network_from_dict(
    G,
    G_type,
    G_color,
    grn_dict_sub,
    figsize=(8,7),
    pos_scale=7,
    TF_size=200,
    node_alpha=0.8,
    node_linewidths=0.5,
    TF_fontsize=12,
    target_fontsize=12,
    specific_gene=top_n_unique_genes(pathway_to_genes,adata_var),
    specific_target_size=100,                  # 指定靶基因节点大小
    other_target_size=5                       # 其他靶基因节点大小
)
plt.title('thalamus GRN',fontsize=15,weight='bold',)
fig.savefig("Figure/whole_brain_selected_GRN/GRN/thalamus_GRN.pdf", dpi=300, bbox_inches = "tight")
plt.show()

# Di_ChP

In [ ]:
adata = sc.read_h5ad("Process_Data/bin100_3D_region/combined_adata_Di_ChP.h5ad")
sc.pp.calculate_qc_metrics(adata, qc_vars=[], inplace=True, log1p=True)
adata_var = adata.var
adata_var['gene'] = adata_var.index
del adata

with open('Output/GRN_GPU_output/Di_ChP/tmp_files/regulons.json', 'r', encoding='utf-8') as file:
    grn_dict = json.load(file)

In [ ]:
keys = ['PAX7(+)','PAX3(+)','MYNN(+)','LMX1A(+)']
grn_dict_sub = {k: grn_dict[k] for k in keys if k in grn_dict}

s = pd.Series(grn_dict_sub)
s_exploded = s.explode()
df = s_exploded.reset_index()
df.columns = ['TF', 'Target Gene']
df.to_csv("Output/whole_brain_selected_GRN/Di_ChP_selected_GRN.csv")
df

In [ ]:
enr=ov.bulk.geneset_enrichment(gene_list=list(set(df['Target Gene'])),
                                pathways_dict=pathway_dict,
                                pvalue_type='auto',
                                organism='Human')

pathway_1 = 'cellular protein localization (GO:0034613)'
pathway_2 = 'cellular protein modification process (GO:0006464)'
pathway_3 = 'regulation of apoptotic process (GO:0042981)'

pathway_to_genes = {pathway_1:[g.strip() for g in list(enr.loc[enr['Term'] == pathway_1, 'Genes'].values)[0].split(';') if g.strip()],
                    pathway_2:[g.strip() for g in list(enr.loc[enr['Term'] == pathway_2, 'Genes'].values)[0].split(';') if g.strip()],
                    pathway_3:[g.strip() for g in list(enr.loc[enr['Term'] == pathway_3, 'Genes'].values)[0].split(';') if g.strip()],}

In [ ]:
target_pathway = pathway_to_genes.keys()
enr = enr[enr['Term'].isin(target_pathway)]

fig,ax = pathway_enrichment(enr.sort_values(['Adjusted P-value'],ascending=True).iloc[0:4,:],color='#FBCAC8',fontsize=15
                                  ,figsize=(4.5,3.5),threshold=40,
                                  title='')
ax.set_xlabel('-log$_{10}$ Adjusted $\it{P}$',fontsize=16,)
plt.grid(False)
ax.tick_params(axis='x', which='major', labelsize=16, width=1.5, length=4,direction='out')
ax.tick_params(axis='y', which='major', labelsize=16, width=1.5, length=0,direction='out')
fig.savefig("Figure/whole_brain_selected_GRN/Pathway/Di_ChP_pathway.pdf", dpi=300, bbox_inches = "tight")
enr.sort_values(['Adjusted P-value']).to_csv("Output/whole_brain_selected_GRN/0_Di_ChP_selected_GRN_pathway.csv")

In [ ]:
import matplotlib.pyplot as plt
G, G_type, G_color = create_network_from_dict(grn_dict_sub, n_top=500,color_dict={'tf_color':'#7030a0','target_color':'#ffc000'})
fig, ax = grn_network_from_dict(
    G,
    G_type,
    G_color,
    grn_dict_sub,
    figsize=(8,7),
    pos_scale=7,
    TF_size=200,
    node_alpha=0.8,
    node_linewidths=0.5,
    TF_fontsize=12,
    target_fontsize=12,
    specific_gene=top10_unique_genes(pathway_to_genes,adata_var),
    specific_target_size=100,                  # 指定靶基因节点大小
    other_target_size=10                       # 其他靶基因节点大小
)
plt.title('Di_ChP GRN',fontsize=15,weight='bold',)
fig.savefig("Figure/whole_brain_selected_GRN/GRN/Di_ChP_GRN.pdf", dpi=300, bbox_inches = "tight")
plt.show()

# Hindbrain_ChP

In [ ]:
adata = sc.read_h5ad("Process_Data/bin100_3D_region/combined_adata_Hindbrain_ChP.h5ad")
sc.pp.calculate_qc_metrics(adata, qc_vars=[], inplace=True, log1p=True)
adata_var = adata.var
adata_var['gene'] = adata_var.index
del adata

with open('Output/GRN_GPU_output/Hindbrain_ChP/tmp_files/regulons.json', 'r', encoding='utf-8') as file:
    grn_dict = json.load(file)

In [ ]:
keys = ['ATF7(+)','BCLAF1(+)','ELF4(+)','HOXA1(+)','HOXB4(+)']
grn_dict_sub = {k: grn_dict[k] for k in keys if k in grn_dict}

s = pd.Series(grn_dict_sub)
s_exploded = s.explode()
df = s_exploded.reset_index()
df.columns = ['TF', 'Target Gene']
df.to_csv("Output/whole_brain_selected_GRN/Hindbrain_ChP_selected_GRN.csv")
df

In [ ]:
enr=ov.bulk.geneset_enrichment(gene_list=list(set(df['Target Gene'])),
                                pathways_dict=pathway_dict,
                                pvalue_type='auto',
                                organism='Human')

pathway_1 = 'cellular protein metabolic process (GO:0044267)'
pathway_2 = 'organelle assembly (GO:0070925)'
pathway_3 = 'cilium assembly (GO:0060271)'
pathway_4 = 'aerobic electron transport chain (GO:0019646)'

pathway_to_genes = {pathway_1:[g.strip() for g in list(enr.loc[enr['Term'] == pathway_1, 'Genes'].values)[0].split(';') if g.strip()],
                    pathway_2:[g.strip() for g in list(enr.loc[enr['Term'] == pathway_2, 'Genes'].values)[0].split(';') if g.strip()],
                    pathway_3:[g.strip() for g in list(enr.loc[enr['Term'] == pathway_3, 'Genes'].values)[0].split(';') if g.strip()],
                    pathway_4:[g.strip() for g in list(enr.loc[enr['Term'] == pathway_4, 'Genes'].values)[0].split(';') if g.strip()],}

In [ ]:
target_pathway = pathway_to_genes.keys()
enr = enr[enr['Term'].isin(target_pathway)]

fig,ax = pathway_enrichment(enr.sort_values(['Adjusted P-value'],ascending=True).iloc[0:4,:],color='#FBCAC8',fontsize=15
                                  ,figsize=(4.5,3.5),threshold=40,
                                  title='')
ax.set_xlabel('-log$_{10}$ Adjusted $\it{P}$',fontsize=16,)
plt.grid(False)
ax.tick_params(axis='x', which='major', labelsize=16, width=1.5, length=4,direction='out')
ax.tick_params(axis='y', which='major', labelsize=16, width=1.5, length=0,direction='out')
fig.savefig("Figure/whole_brain_selected_GRN/Pathway/Hindbrain_ChP_pathway.pdf", dpi=300, bbox_inches = "tight")
enr.sort_values(['Adjusted P-value']).to_csv("Output/whole_brain_selected_GRN/0_Hindbrain_ChP_selected_GRN_pathway.csv")

In [ ]:
import matplotlib.pyplot as plt
G, G_type, G_color = create_network_from_dict(grn_dict_sub, n_top=5000,color_dict={'tf_color':'#7030a0','target_color':'#ffc000'})
fig, ax = grn_network_from_dict(
    G,
    G_type,
    G_color,
    grn_dict_sub,
    figsize=(8,7),
    pos_scale=7,
    TF_size=200,
    node_alpha=0.8,
    node_linewidths=0.5,
    TF_fontsize=12,
    target_fontsize=12,
    specific_gene=top10_unique_genes(pathway_to_genes,adata_var),
    specific_target_size=100,                  # 指定靶基因节点大小
    other_target_size=5                       # 其他靶基因节点大小
)
plt.title('Hindbrain_ChP GRN',fontsize=15,weight='bold',)
fig.savefig("Figure/whole_brain_selected_GRN/GRN/Hindbrain_ChP_GRN.pdf", dpi=300, bbox_inches = "tight")
plt.show()